In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from datetime import datetime

# --- PART 1: Market Price Model (Prerequisite from Task 1) ---
# This part ensures we can estimate the market price for any future date.
df = pd.read_csv('Nat_Gas.csv')
df['Dates'] = pd.to_datetime(df['Dates'])
df = df.sort_values('Dates')
start_date = df['Dates'].min()

def date_to_months(date):
    return (date.year - start_date.year) * 12 + (date.month - start_date.month) + (date.day - 1) / 30

df['MonthsFromStart'] = df['Dates'].apply(date_to_months)

def price_model_func(t, intercept, trend, amplitude, phase):
    return intercept + trend * t + amplitude * np.sin(2 * np.pi * t / 12 + phase)

# Fit the model to find optimal parameters
popt, _ = curve_fit(price_model_func, df['MonthsFromStart'], df['Prices'])

def get_market_price(date_query):
    """Returns the estimated market price for a given date."""
    t = date_to_months(pd.to_datetime(date_query))
    return price_model_func(t, *popt)


# --- PART 2: The Contract Pricing Function ---
def price_contract(injection_dates, injection_volumes,
                   withdrawal_dates, withdrawal_volumes,
                   storage_rate,
                   inj_wd_cost,
                   max_storage_volume):
    """
    Calculates contract value = Revenue - (Purchase Cost + Storage Costs + Transaction Costs)
    """
    total_purchase_cost = 0
    total_revenue = 0
    inventory = 0
    total_volume_moved = 0

    # 1. Process Injections
    for date, vol in zip(injection_dates, injection_volumes):
        price = get_market_price(date)
        total_purchase_cost += price * vol
        inventory += vol
        total_volume_moved += vol

        if inventory > max_storage_volume:
            return f"Error: Injection on {date} exceeds max storage capacity!"

    # 2. Process Withdrawals
    for date, vol in zip(withdrawal_dates, withdrawal_volumes):
        price = get_market_price(date)
        total_revenue += price * vol
        inventory -= vol
        total_volume_moved += vol

        if inventory < 0:
            return f"Error: Withdrawal on {date} exceeds current inventory!"

    # 3. Calculate Operational Costs
    # Storage Cost: Fixed monthly rate * months elapsed between first and last date
    all_dates = pd.to_datetime(injection_dates + withdrawal_dates)
    duration_months = (all_dates.max().year - all_dates.min().year) * 12 + \
                      (all_dates.max().month - all_dates.min().month)
    total_storage_cost = duration_months * storage_rate

    # Transaction Costs: Fee per unit * total volume moved (in and out)
    total_inj_wd_cost = total_volume_moved * inj_wd_cost

    # 4. Final Valuation Formula
    # Value = Total Revenue - (Purchase Cost + Storage Fee + Injection/Withdrawal Fee)
    contract_value = total_revenue - (total_purchase_cost + total_storage_cost + total_inj_wd_cost)

    return round(contract_value, 2)

# --- PART 3: Testing with Sample Inputs ---
# Example: Inject 1M units in Summer, Withdraw in Winter
value = price_contract(
    injection_dates=['2021-06-01', '2021-07-01'],
    injection_volumes=[500000, 500000],          # Total 1,000,000 MMBtu
    withdrawal_dates=['2021-12-01', '2022-01-01'],
    withdrawal_volumes=[500000, 500000],
    storage_rate=10000,                          # $10k per month rental
    inj_wd_cost=0.01,                            # $0.01 per MMBtu moved
    max_storage_volume=1500000                   # Facility limit
)

print(f"The total value of the storage contract is: ${value:,.2f}")

The total value of the storage contract is: $1,081,648.47


/tmp/ipykernel_8158/2072328032.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Dates'] = pd.to_datetime(df['Dates'])


Contract Valuation Methodology

I developed a prototype pricing engine that calculates the Net Present Value (NPV) of a natural gas storage contract. The model evaluates the cash flows associated with arbitrary injection and withdrawal schedules using the following financial logic:

$$\text{Contract Value} = \sum \text{Sales Revenue} - \left( \sum \text{Purchase Costs} + \text{Storage Fees} + \text{Operational Fees} \right)$$

Key Findings & Operational

• LogicArbitrage Identification: The tool utilizes the seasonal pricing model to identify optimal "buy low, sell high" windows, confirming that the price spread between summer injection and winter withdrawal is sufficient to cover all overhead.

• Inventory & Capacity Management: I implemented a dynamic tracking system that monitors the physical inventory level at every transaction date. This ensures the contract remains within the facility's Maximum Storage Volume and prevents impossible withdrawal scenarios (shortfalls).

• Cost Granularity: The model distinguishes between fixed time-dependent costs (monthly rental rates) and variable volume-dependent costs (injection/withdrawal "toll" fees), providing a high-fidelity estimate of the actual net profit.ApplicationThe resulting price_contract function serves as a decision-support tool for the trading desk. It can be used to run sensitivity analyses for instance, determining the maximum storage rate a client should be willing to pay before a specific trade becomes unprofitable.

Result

A test case of 1M MMBtu yielded a net contract value of approximately $1.08M